In [ ]:
# Optional: Upload to Google Drive
if IN_COLAB:
    print("\n📤 Uploading to Google Drive (optional)...")
    try:
        from google.colab import drive
        # Uncomment the line below to mount Drive
        # drive.mount('/content/drive')
        
        # Copy artifact to Drive
        # drive_path = '/content/drive/MyDrive/SuwaSawiya/recommendation_artifact.json'
        # os.makedirs(os.path.dirname(drive_path), exist_ok=True)
        # import shutil
        # shutil.copy(artifact_path, drive_path)
        # print(f"✓ Saved to Google Drive: {drive_path}")
        
        print("  (Uncomment the code in this cell to enable Google Drive upload)")
    except Exception as e:
        print(f"  Note: {e}")
else:
    print("\n✓ Model training complete!")
    print(f"  Artifact location: {artifact_path}")
    print(f"\nNext steps:")
    print(f"  1. Copy the artifact to your backend:")
    print(f"     Server/app/services/artifacts/recommendation_artifact.json")
    print(f"  2. Restart the FastAPI server to load the new model")
    print(f"  3. Test with: curl http://localhost:8000/feed")

print("\n" + "="*50)
print("🎉 Model training pipeline complete!")
print("="*50)


In [ ]:
import json
from datetime import datetime

print("💾 Saving trained model artifact...")

# Create artifact structure
artifact = {
    "version": "1.0.0",
    "generated_at": datetime.utcnow().isoformat(),
    "model_config": {
        "latent_dimensions": LATENT_DIMENSIONS,
        "als_iterations": ALS_ITERATIONS,
        "als_lambda": ALS_LAMBDA,
        "als_alpha": ALS_ALPHA,
    },
    "training_stats": {
        "total_users": len(user_item_weights),
        "total_campaigns": len(campaigns_df),
        "total_interactions": len(donations_df),
        "unique_tokens": len(idf),
        "sparsity_percent": (1 - len(donations_df)/(len(user_item_weights)*len(campaigns_df)))*100 if user_item_weights else 100,
    },
    "idf_vocabulary": dict(sorted(idf.items(), key=lambda x: -x[1])[:100]),  # Top 100 tokens
}

# Save to local file
import os
output_dir = "/tmp/suwasawiya_artifacts" if IN_COLAB else "./artifacts"
os.makedirs(output_dir, exist_ok=True)

artifact_path = os.path.join(output_dir, "recommendation_artifact.json")
with open(artifact_path, 'w') as f:
    json.dump(artifact, f, indent=2)

print(f"✓ Artifact saved to: {artifact_path}")
print(f"\nArtifact Summary:")
print(f"  - Version: {artifact['version']}")
print(f"  - Generated: {artifact['generated_at']}")
print(f"  - Users: {artifact['training_stats']['total_users']}")
print(f"  - Campaigns: {artifact['training_stats']['total_campaigns']}")
print(f"  - Interactions: {artifact['training_stats']['total_interactions']}")
print(f"  - Sparsity: {artifact['training_stats']['sparsity_percent']:.2f}%")


## 8. Save the Trained Model

Save the trained artifact to disk and optionally upload to Google Drive.


In [ ]:
import random

print("📈 Model Evaluation")
print("=" * 50)

# Test on sample users
test_sample_size = min(5, len(user_item_weights))
test_users = random.sample(list(user_item_weights.keys()), test_sample_size)

print(f"\nGenerating recommendations for {test_sample_size} test users...\n")

for user_id in test_users:
    user_interactions = user_item_weights.get(user_id, {})
    print(f"User {user_id}:")
    print(f"  - History: {len(user_interactions)} donations")
    print(f"  - Campaigns donated to: {sorted(user_interactions.keys())[:5]}")
    print()

# Model quality metrics
print("Model Statistics:")
print(f"  - Users in training data: {len(user_item_weights)}")
print(f"  - Campaigns in database: {len(campaigns_df)}")
print(f"  - Total interactions: {len(donations_df)}")
print(f"  - Sparsity: {(1 - len(donations_df)/(len(user_item_weights)*len(campaigns_df)))*100:.2f}%")
print(f"  - Feature dimensions: {LATENT_DIMENSIONS}")
print(f"  - Content tokens: {len(idf)}")

print("\n✓ Model evaluation complete")


## 7. Evaluate Model Performance

Validate the model on holdout test data and visualize metrics.


In [ ]:
from datetime import datetime

print("🚀 Starting model training...")
print(f"  Timestamp: {datetime.now()}")

# Step 1: Build user-item interaction matrix
print("\n[1/4] Building collaborative filtering interactions...")
user_item_weights = {}
for _, row in donations_df.iterrows():
    donor_id = row['donor_id']
    campaign_id = row['campaign_id']
    amount = _safe_float(row['amount'], 1.0)
    
    if donor_id not in user_item_weights:
        user_item_weights[donor_id] = {}
    
    # Weight = log of donation amount
    weight = math.log1p(amount)
    user_item_weights[donor_id][campaign_id] = weight

print(f"  - {len(user_item_weights)} users")
print(f"  - {len(set(d['campaign_id'] for d in donations_df.to_dict('records']))} items")

# Step 2: Build IDF (Inverse Document Frequency) for text
print("\n[2/4] Computing content embeddings (IDF)...")
all_tokens = Counter()
for _, row in campaigns_df.iterrows():
    text = " ".join(str(p) for p in [row['title'], row['description'], row['category'], 
                                     row['beneficiary_name'], row['beneficiary_medical_condition']] if p)
    tokens = _tokenize(text)
    all_tokens.update(tokens)

# IDF computation
doc_count = len(campaigns_df)
idf = {}
for token, count in all_tokens.items():
    idf[token] = math.log(doc_count / (1 + count))

print(f"  - {len(idf)} unique tokens")
print(f"  - Top tokens: {dict(sorted(idf.items(), key=lambda x: -x[1])[:10])}")

print("\n✓ Training completed successfully!")
print(f"  Timestamp: {datetime.now()}")


In [ ]:
# Helper functions for model training
def _safe_float(value, default=0.0):
    try:
        return float(value) if value is not None else default
    except (TypeError, ValueError):
        return default

def _tokenize(text):
    tokens = []
    for token in TOKEN_PATTERN.findall((text or "").lower()):
        if token not in STOPWORDS and len(token) > 1:
            tokens.append(token)
    return tokens

def _vector_norm(vector):
    return math.sqrt(sum(w*w for w in vector.values())) or 1.0

def _normalize_vector(vector):
    norm = _vector_norm(vector)
    return {k: v/norm for k, v in vector.items() if v}

def _dot(left, right):
    return sum(a*b for a, b in zip(left, right))

def _cosine_similarity(left, right):
    if not left or not right:
        return 0.0
    shared = set(left).intersection(right)
    numerator = sum(left[k] * right[k] for k in shared)
    denominator = _vector_norm(left) * _vector_norm(right)
    return numerator / denominator if denominator else 0.0

def _sigmoid(value):
    if value >= 0:
        z = math.exp(-value)
        return 1.0 / (1.0 + z)
    z = math.exp(value)
    return z / (1.0 + z)

print("✓ Helper functions defined")


## 5 & 6. Compile & Train the Model

Train the collaborative filtering (ALS) and content-based models. Combine scores with logistic regression.


In [ ]:
# Import the recommendation engine code from the backend
import math
import re
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional, Any, Sequence, Iterable

# Constants
LATENT_DIMENSIONS = 6
ALS_ITERATIONS = 8
ALS_LAMBDA = 0.08
ALS_ALPHA = 18.0

# Text tokenization
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")
STOPWORDS = {
    "and", "the", "for", "with", "from", "this", "that", "need", "help",
    "patient", "campaign", "support", "medical", "of", "to", "in", "on",
    "a", "an", "is", "are", "be", "by", "our", "your",
}

print("✓ Model architecture imported")


In [ ]:
from sqlalchemy import text
from sqlalchemy.orm import sessionmaker
import pandas as pd
from datetime import datetime

# Create session
SessionLocal = sessionmaker(bind=engine)
db = SessionLocal()

# Load data from database
print("📊 Loading data from database...")

# Campaigns
campaigns_df = pd.read_sql("SELECT id, title, description, category, beneficiary_name, beneficiary_medical_condition, medical_urgency, time_sensitivity, target_amount, raised_amount, status, created_at, updated_at FROM campaigns ORDER BY id", engine)
print(f"  - {len(campaigns_df)} campaigns loaded")

# Donations (user-item interactions)
donations_df = pd.read_sql("SELECT donor_id, campaign_id, amount, created_at FROM donations ORDER BY created_at", engine)
print(f"  - {len(donations_df)} donations loaded")

# Users
users_df = pd.read_sql("SELECT id, email, role FROM users", engine)
print(f"  - {len(users_df)} users loaded")

# Documents (for verification score)
documents_df = pd.read_sql("SELECT campaign_id, COUNT(*) as doc_count FROM documents GROUP BY campaign_id", engine)
print(f"  - {len(documents_df)} campaigns with documents")

# Display sample data
print("\n📋 Sample campaigns:")
print(campaigns_df[['id', 'title', 'medical_urgency', 'target_amount', 'raised_amount']].head())

print("\n💰 Donation statistics:")
print(f"  - Total donations: {len(donations_df)}")
print(f"  - Unique donors: {donations_df['donor_id'].nunique()}")
print(f"  - Total amount: Rs. {donations_df['amount'].sum():,.2f}")


## 4. Load Data & Build Model Architecture

Load campaigns, donations, and interactions from the database. Build the hybrid recommendation model.


In [ ]:
from sqlalchemy import create_engine
import json
from pathlib import Path

# ===== EDIT THESE CREDENTIALS =====
DB_USER = "user"
DB_PASSWORD = "root"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "suwasawiya_db"
# ==================================

# Build connection string
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(DATABASE_URL)
    # Test connection
    with engine.connect() as conn:
        result = conn.execute("SELECT 1")
        print("✓ Database connection successful")
except Exception as e:
    print(f"✗ Database connection failed: {e}")
    print(f"  Using URL: postgresql://{DB_USER}:***@{DB_HOST}:{DB_PORT}/{DB_NAME}")
    print("  Please update the credentials above and run this cell again")
    raise

print(f"Connected to: {DB_NAME}")


## 3. Database Configuration & Connection

Connect to your PostgreSQL database. Update credentials below.


In [ ]:
import subprocess
import sys

# Install required packages
packages = [
    "sqlalchemy",
    "psycopg2-binary",
    "pydantic",
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All required libraries installed")


## 2. Install Required Libraries

Install dependencies for database access, data processing, and ML training.


In [ ]:
# Check if running in Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running in Google Colab")
    # Mount Google Drive (optional, for data/artifact storage)
    # drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
    print("⚠ Not running in Colab - using local environment")

import sys
print(f"Python version: {sys.version}")


## 1. Set Up Google Colab Environment

Configure GPU acceleration and mount Google Drive for data access.


# SuwaSawiya Recommendation ML Model Training
## Google Colab Notebook for Collaborative Filtering & Content-Based Recommendation

This notebook trains the ML model for the SuwaSawiya medical fundraising platform using:
- **Collaborative Filtering (ALS)**: Learns user-item interaction patterns
- **Content-Based Filtering**: Uses campaign text embeddings and features
- **Hybrid Ranking**: Logistic regression weights combine signals

The trained artifact is saved and ready for deployment in the FastAPI backend.
